# W2 Day5 (04/18 周五): Diffusion 原理 ★

## 学习目标
- **理论 (1-1.5h)**: DDPM 前向加噪 / 反向去噪 (直觉理解，不需完整 ELBO 推导); DDIM 加速; Classifier-Free Guidance; ControlNet 架构
- **实践 (3h)**: 跑 Stable Diffusion pipeline + ControlNet 条件生成 + FID/CLIP Score 评估
- **产出物**: Diffusion 实验 notebook (★ 合成数据核心)

## 为什么这天对你最重要
你的博士论文 (AiC under revision) 核心就是「本体引导文生图合成训练数据」。今天你要把过去用过的 Diffusion 工具**从工程使用者升级为能讲清楚原理的人**:
- 「为什么 ControlNet 能控制结构而不破坏预训练权重？」← 面试官真的会问
- 「合成数据有 domain gap 你怎么办？」← 你的论文方法论
- 「FID 和 CLIP Score 各衡量什么？什么时候哪个失效？」← 评估指标硬核

## 核心问题 (面试高频)
1. DDPM 训练目标是预测什么？为什么是噪声 ε 而不是 x_0？
2. DDIM 为什么能减少采样步数？牺牲了什么？
3. CFG 的 guidance scale 调大调小有什么影响？
4. ControlNet 的「锁定权重 + 零卷积」设计为什么 work？
5. FID 在小数据集上为什么不可靠？

---

## 目录
1. [生成模型景观](#1)
2. [DDPM 前向: 加噪过程](#2)
3. [DDPM 反向: 训练目标](#3)
4. [小型 DDPM 实战 (MNIST)](#4)
5. [DDIM 加速采样](#5)
6. [Classifier-Free Guidance](#6)
7. [Stable Diffusion 架构](#7)
8. [ControlNet 原理](#8)
9. [评估指标: FID & CLIP Score](#9)
10. [总结 & 思考题](#10)


---
## 1. 生成模型景观 <a id='1'></a>

### 1.1 主流生成模型对比

| 模型 | 训练目标 | 生成速度 | 模式覆盖 | 可控性 |
|---|---|---|---|---|
| **VAE** | ELBO (重构 + KL) | 快 (1 步) | 好 | 中 |
| **GAN** | 对抗博弈 | 快 (1 步) | 易 mode collapse | 中 |
| **Flow** | 精确 log-likelihood | 中 | 好 | 中 |
| **Diffusion** | 噪声预测 (简单 MSE) | **慢 (~50 步)** | **极好** | **极好** |
| **Autoregressive** | next-token | 慢 (序列长) | 极好 | 强 |

### 1.2 Diffusion 为什么赢了？

1. **训练稳定**: MSE loss，没有 GAN 的对抗博弈
2. **模式覆盖**: 不会 mode collapse (因为是 likelihood-based)
3. **生成质量**: SOTA (FID 大幅领先 GAN)
4. **可控性**: ControlNet / IP-Adapter / LoRA 等扩展生态成熟

代价: 推理慢 (DDIM 仍需 ~20-50 步)。这就是为什么有 Latent Diffusion (在小空间扩散) 和 Distillation (LCM, Turbo, SDXL Lightning)。

### 1.3 直觉: 把生成问题转化为去噪问题

```
真实图像 x_0 ─加噪─→ x_T (纯噪声)
                ↑
         反向: 学习去噪
                ↓
            从噪声采样 → 一步步去噪 → 生成图像
```

关键 insight: **去噪比直接生成简单**。神经网络只需要学习「这张有点模糊的图像应该长什么样」，而不是「无中生有一张图像」。


In [ ]:
import math
import time
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")


---
## 2. DDPM 前向: 加噪过程 <a id='2'></a>

### 2.1 前向过程定义

给定真实图像 $x_0$，定义 T 步加噪 (通常 T=1000):

$$q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1-\beta_t} \cdot x_{t-1}, \beta_t I)$$

其中 $\beta_t$ 是噪声调度 (linear/cosine)，从 1e-4 慢慢增大到 0.02。

### 2.2 关键: 一步到位公式

定义 $\alpha_t = 1 - \beta_t$, $\bar\alpha_t = \prod_{s=1}^{t} \alpha_s$，**可以直接从 $x_0$ 一步算到 $x_t$**:

$$x_t = \sqrt{\bar\alpha_t} \cdot x_0 + \sqrt{1-\bar\alpha_t} \cdot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

这是 DDPM 的**核心**:
- 训练时不需要逐步加噪 → 任意时间步 t 直接采样
- $\bar\alpha_t \to 0$ 时 (t 大)，$x_t \approx \epsilon$ (纯噪声)
- $\bar\alpha_t \to 1$ 时 (t 小)，$x_t \approx x_0$ (原图)

### 2.3 噪声调度: linear vs cosine

- **Linear** (DDPM 原版): $\beta_t$ 线性从 1e-4 到 0.02
  - 简单，但前期信息损失太快
- **Cosine** (Improved DDPM): 余弦曲线
  - 信息损失更平缓，效果更好


In [ ]:
def linear_beta_schedule(T, beta_start=1e-4, beta_end=0.02):
    return torch.linspace(beta_start, beta_end, T)


def cosine_beta_schedule(T, s=0.008):
    """Improved DDPM 提出的 cosine 调度"""
    steps = T + 1
    x = torch.linspace(0, T, steps)
    alphas_bar = torch.cos(((x / T) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_bar = alphas_bar / alphas_bar[0]
    betas = 1 - (alphas_bar[1:] / alphas_bar[:-1])
    return torch.clip(betas, 0.0001, 0.9999)


T = 1000
beta_lin = linear_beta_schedule(T)
beta_cos = cosine_beta_schedule(T)

alpha_lin = 1 - beta_lin
alpha_bar_lin = torch.cumprod(alpha_lin, dim=0)
alpha_cos = 1 - beta_cos
alpha_bar_cos = torch.cumprod(alpha_cos, dim=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(beta_lin, label='linear', color='#1976D2')
axes[0].plot(beta_cos, label='cosine', color='#E64A19')
axes[0].set_title(r'$\beta_t$ schedule')
axes[0].set_xlabel('t')
axes[0].set_ylabel(r'$\beta_t$')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(alpha_bar_lin, label='linear', color='#1976D2')
axes[1].plot(alpha_bar_cos, label='cosine', color='#E64A19')
axes[1].set_title(r'$\bar\alpha_t$: 信号保留比例')
axes[1].set_xlabel('t')
axes[1].set_ylabel(r'$\bar\alpha_t$')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nLinear:  α_bar 在 t=200 已降到 {alpha_bar_lin[200]:.3f} (信息损失过快)")
print(f"Cosine:  α_bar 在 t=200 还有 {alpha_bar_cos[200]:.3f} (更平缓)")


In [ ]:
# 可视化前向加噪过程: 用一张简单的合成图像
def make_synthetic_image(size=32):
    """生成一张简单的几何图像"""
    img = torch.zeros(1, 1, size, size)
    img[0, 0, 8:24, 8:24] = 1.0  # 中心方块
    img[0, 0, 12:20, 12:20] = 0.0  # 中心挖空
    img[0, 0, 14:18, 14:18] = 0.5  # 内部小方块
    return img * 2 - 1  # 归一化到 [-1, 1] (DDPM 标准)


x_0 = make_synthetic_image(32)
betas = linear_beta_schedule(T)
alphas = 1 - betas
alphas_bar = torch.cumprod(alphas, dim=0)


def q_sample(x_0, t, alphas_bar, noise=None):
    """前向: 一步到位采样 x_t"""
    if noise is None:
        noise = torch.randn_like(x_0)
    sqrt_alpha_bar = alphas_bar[t].sqrt()
    sqrt_one_minus_alpha_bar = (1 - alphas_bar[t]).sqrt()
    return sqrt_alpha_bar * x_0 + sqrt_one_minus_alpha_bar * noise


# 在多个时间步可视化
ts = [0, 50, 100, 250, 500, 750, 999]
fig, axes = plt.subplots(1, len(ts), figsize=(14, 2.5))
torch.manual_seed(42)
noise = torch.randn_like(x_0)
for ax, t in zip(axes, ts):
    if t == 0:
        x_t = x_0
    else:
        x_t = q_sample(x_0, t, alphas_bar, noise)
    ax.imshow(x_t[0, 0].numpy(), cmap='gray', vmin=-1, vmax=1)
    ax.set_title(f't={t}\nα̅={alphas_bar[t].item():.3f}', fontsize=10)
    ax.axis('off')
plt.suptitle('DDPM 前向过程: 不同时间步的 $x_t$', y=1.05)
plt.tight_layout()
plt.show()

print("\n💡 t=999 时图像几乎纯噪声 (α̅≈0)")
print("💡 训练时随机采样 t∈[0,T]，让模型学会去噪不同强度的噪声")


---
## 3. DDPM 反向: 训练目标 <a id='3'></a>

### 3.1 反向过程

我们想学的是 $p_\theta(x_{t-1} | x_t)$ — 从有噪图像还原少噪图像。

**关键洞察**: 如果 t 很小 (β_t 很小)，反向过程近似为高斯分布:

$$p_\theta(x_{t-1} | x_t) = \mathcal{N}(x_{t-1}; \mu_\theta(x_t, t), \Sigma_\theta(x_t, t))$$

### 3.2 简化训练目标 (DDPM 论文核心贡献)

经过 ELBO 推导和重参数化技巧 (这里跳过推导)，训练目标简化为:

$$L_{\text{simple}} = \mathbb{E}_{t, x_0, \epsilon} \left[ \| \epsilon - \epsilon_\theta(x_t, t) \|^2 \right]$$

**人话**: 给模型一张加噪图 $x_t$ 和时间步 $t$，让它**预测加进去的噪声 $\epsilon$**。

为什么不直接预测 $x_0$？两者等价 (因为知道 $x_t$ 和 $\epsilon$ 就能算 $x_0$)，但实验表明**预测 $\epsilon$ 更稳定** (因为 ε 始终是标准正态，量纲不变)。

### 3.3 训练循环 (伪代码)

```python
for x_0 in dataloader:
    t = randint(0, T)              # 随机时间步
    eps = randn_like(x_0)          # 随机噪声
    x_t = sqrt(α̅[t]) * x_0 + sqrt(1-α̅[t]) * eps  # 前向加噪
    eps_pred = model(x_t, t)       # 模型预测噪声
    loss = MSE(eps, eps_pred)      # 简单 MSE
    loss.backward()
    optimizer.step()
```

简单到难以置信，这就是 DDPM 的优雅之处。

### 3.4 反向采样 (生成时)

从 $x_T \sim \mathcal{N}(0, I)$ 出发，逐步去噪 T 步:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{1-\alpha_t}{\sqrt{1-\bar\alpha_t}} \epsilon_\theta(x_t, t) \right) + \sigma_t z, \quad z \sim \mathcal{N}(0,I)$$

其中 $\sigma_t = \sqrt{\beta_t}$ (DDPM 的随机性来源)。


---
## 4. 小型 DDPM 实战 (MNIST) <a id='4'></a>

我们用一个**极简 U-Net** 在 MNIST 上训练一个真正的 DDPM，体会:
1. 模型只是预测噪声 (input: x_t + t, output: ε)
2. 训练 loss 是简单的 MSE
3. 采样时从纯噪声逐步去噪

为了能在 CPU 也能跑，我们:
- 只用 100 个数字 0 训练
- 模型很小 (~50K 参数)
- 200 iter 训练
- 实际只是为了**演示流程**，质量不会太好


In [ ]:
class TinyUNet(nn.Module):
    """
    极简 U-Net for DDPM (MNIST 28×28)
    
    输入: x (B, 1, 28, 28), t (B,)
    输出: ε_pred (B, 1, 28, 28)
    """
    def __init__(self, time_dim=64, hidden=32):
        super().__init__()
        self.time_dim = time_dim
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, hidden * 4),
            nn.SiLU(),
            nn.Linear(hidden * 4, hidden * 4),
        )
        # Down
        self.conv1 = nn.Conv2d(1, hidden, 3, padding=1)
        self.conv2 = nn.Conv2d(hidden, hidden * 2, 3, padding=1, stride=2)  # 14×14
        self.conv3 = nn.Conv2d(hidden * 2, hidden * 4, 3, padding=1, stride=2)  # 7×7
        # Up
        self.up1 = nn.ConvTranspose2d(hidden * 4, hidden * 2, 4, stride=2, padding=1)  # 14×14
        self.up2 = nn.ConvTranspose2d(hidden * 4, hidden, 4, stride=2, padding=1)      # 28×28 (skip)
        self.out = nn.Conv2d(hidden * 2, 1, 3, padding=1)
        # Time injection
        self.time_proj1 = nn.Linear(hidden * 4, hidden)
        self.time_proj2 = nn.Linear(hidden * 4, hidden * 2)
        self.time_proj3 = nn.Linear(hidden * 4, hidden * 4)
    
    @staticmethod
    def sinusoidal_embedding(t, dim):
        """位置编码风格的时间步嵌入"""
        half = dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        args = t[:, None].float() * freqs[None, :]
        return torch.cat([args.sin(), args.cos()], dim=-1)
    
    def forward(self, x, t):
        t_emb = self.sinusoidal_embedding(t, self.time_dim)
        t_emb = self.time_mlp(t_emb)  # (B, hidden*4)
        
        # Down
        h1 = F.silu(self.conv1(x) + self.time_proj1(t_emb)[:, :, None, None])  # (B, h, 28, 28)
        h2 = F.silu(self.conv2(h1) + self.time_proj2(t_emb)[:, :, None, None])  # (B, 2h, 14, 14)
        h3 = F.silu(self.conv3(h2) + self.time_proj3(t_emb)[:, :, None, None])  # (B, 4h, 7, 7)
        
        # Up + skip
        u1 = F.silu(self.up1(h3))                          # (B, 2h, 14, 14)
        u1 = torch.cat([u1, h2], dim=1)                    # (B, 4h, 14, 14)
        u2 = F.silu(self.up2(u1))                          # (B, h, 28, 28)
        u2 = torch.cat([u2, h1], dim=1)                    # (B, 2h, 28, 28)
        
        return self.out(u2)


unet = TinyUNet().to(device)
print(f"TinyUNet 参数量: {sum(p.numel() for p in unet.parameters()):,}")

# 测试前向
x_test = torch.randn(2, 1, 28, 28).to(device)
t_test = torch.tensor([100, 500]).to(device)
out = unet(x_test, t_test)
print(f"前向: x{tuple(x_test.shape)}, t{tuple(t_test.shape)} → {tuple(out.shape)}")


In [ ]:
# 准备一个小数据集 (合成的简单图案，避免下载 MNIST 的网络依赖)
def make_simple_dataset(n=200, size=28):
    """生成简单图案: 圆 / 方 / 三角 各占 1/3"""
    imgs = []
    for i in range(n):
        img = torch.zeros(1, size, size)
        kind = i % 3
        if kind == 0:  # 圆
            cx, cy = size // 2, size // 2
            r = 8
            for x in range(size):
                for y in range(size):
                    if (x-cx)**2 + (y-cy)**2 <= r**2:
                        img[0, x, y] = 1.0
        elif kind == 1:  # 方
            img[0, 8:20, 8:20] = 1.0
        else:  # 三角 (近似)
            for r in range(8, 22):
                w = (r - 8) // 2
                img[0, r, 14-w:14+w+1] = 1.0
        # 加一点抖动
        img += torch.randn_like(img) * 0.05
        img = img.clamp(0, 1)
        imgs.append(img)
    
    data = torch.stack(imgs) * 2 - 1  # 归一化到 [-1, 1]
    return data


dataset = make_simple_dataset(200, 28)
print(f"Dataset: {tuple(dataset.shape)}")

# 显示样本
fig, axes = plt.subplots(1, 6, figsize=(12, 2))
for ax, i in zip(axes, [0, 1, 2, 30, 31, 32]):
    ax.imshow(dataset[i, 0].numpy(), cmap='gray', vmin=-1, vmax=1)
    ax.axis('off')
plt.suptitle('合成训练数据 (圆/方/三角)')
plt.tight_layout()
plt.show()


In [ ]:
# DDPM 训练
T_train = 200  # 用更小的 T 加速教学
betas = linear_beta_schedule(T_train, 1e-4, 0.02).to(device)
alphas = (1 - betas).to(device)
alphas_bar = torch.cumprod(alphas, dim=0)
sqrt_alphas_bar = alphas_bar.sqrt()
sqrt_one_minus_alphas_bar = (1 - alphas_bar).sqrt()

unet = TinyUNet(hidden=32).to(device)
optimizer = torch.optim.AdamW(unet.parameters(), lr=2e-4)

dataset = dataset.to(device)
batch_size = 32
n_iters = 800

losses = []
t0 = time.time()
print(f"训练 DDPM ({n_iters} iter)...")

for it in range(n_iters):
    # 随机 batch
    idx = torch.randint(0, len(dataset), (batch_size,))
    x_0 = dataset[idx]
    
    # 随机时间步 + 随机噪声
    t = torch.randint(0, T_train, (batch_size,), device=device)
    eps = torch.randn_like(x_0)
    
    # 前向加噪
    x_t = sqrt_alphas_bar[t][:, None, None, None] * x_0 + \
          sqrt_one_minus_alphas_bar[t][:, None, None, None] * eps
    
    # 模型预测噪声
    eps_pred = unet(x_t, t)
    
    # MSE loss
    loss = F.mse_loss(eps_pred, eps)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    if (it + 1) % 100 == 0:
        avg = np.mean(losses[-50:])
        print(f"  iter {it+1:4d}: loss = {avg:.4f}")

print(f"训练完成: {time.time()-t0:.1f}s")

plt.figure(figsize=(8, 3))
plt.plot(losses, alpha=0.4)
# 滑窗平均
w = 30
smooth = np.convolve(losses, np.ones(w)/w, mode='valid')
plt.plot(range(w-1, len(losses)), smooth, color='red', label=f'MA-{w}')
plt.xlabel('iteration'); plt.ylabel('MSE loss')
plt.title('DDPM 训练 loss (噪声预测 MSE)')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
@torch.no_grad()
def ddpm_sample(model, n_samples, image_size=28, T=T_train):
    """DDPM 采样: 从纯噪声逐步去噪 T 步"""
    model.eval()
    x = torch.randn(n_samples, 1, image_size, image_size).to(device)
    
    history = [x.cpu().clone()]  # 记录中间步
    save_steps = [T-1, int(T*0.75), int(T*0.5), int(T*0.25), 0]
    
    for t in reversed(range(T)):
        t_batch = torch.full((n_samples,), t, device=device, dtype=torch.long)
        eps_pred = model(x, t_batch)
        
        alpha_t = alphas[t]
        alpha_bar_t = alphas_bar[t]
        beta_t = betas[t]
        
        # 反向均值
        x_mean = (1 / alpha_t.sqrt()) * (x - (beta_t / (1 - alpha_bar_t).sqrt()) * eps_pred)
        
        if t > 0:
            noise = torch.randn_like(x)
            x = x_mean + beta_t.sqrt() * noise
        else:
            x = x_mean
        
        if t in save_steps:
            history.append(x.cpu().clone())
    
    model.train()
    return x, history


x_gen, history = ddpm_sample(unet, n_samples=6)
print(f"生成完成: {tuple(x_gen.shape)}")

# 可视化生成过程: 每个 sample 的去噪轨迹
fig, axes = plt.subplots(6, len(history), figsize=(len(history)*1.2, 7))
for i in range(6):
    for j, x_step in enumerate(history):
        axes[i, j].imshow(x_step[i, 0].numpy(), cmap='gray', vmin=-1, vmax=1)
        axes[i, j].axis('off')
        if i == 0:
            axes[i, j].set_title(f'step {len(history)-1-j}')
plt.suptitle('DDPM 反向去噪过程: T → 0', y=1.01)
plt.tight_layout()
plt.show()

print("\n💡 数据规模和训练时间不够 → 生成质量有限，但能看到从纯噪声逐步形成结构")
print("💡 真实 SD 训练用 5亿+ 图像 + 上千 GPU 天")


---
## 5. DDIM 加速采样 <a id='5'></a>

### 5.1 问题: DDPM 采样太慢

DDPM 训练时用 T=1000，采样也要 1000 步。每步要前向一次模型 → 生成一张图要 ~30 秒 (即使 4090)。

### 5.2 DDIM 的关键洞察

DDIM 论文证明: **训练目标不变**的情况下，采样可以是**确定性的非马尔可夫过程**:

$$x_{t-1} = \sqrt{\bar\alpha_{t-1}} \cdot \hat x_0 + \sqrt{1 - \bar\alpha_{t-1}} \cdot \epsilon_\theta(x_t, t)$$

其中 $\hat x_0 = (x_t - \sqrt{1-\bar\alpha_t} \epsilon_\theta) / \sqrt{\bar\alpha_t}$ 是预测的 $x_0$。

**特性**:
- 确定性: 没有随机噪声项 → 同样的 $x_T$ 始终生成同样的 $x_0$
- 可跳步: 不再要求逐步去噪，可以跳着采样 (例如 1000 → 500 → ... → 0)
- **同样的训练好的模型，可以用 50 步出图，质量损失很小**

### 5.3 DDIM vs DDPM 实测

- DDIM 50 步 ≈ DDPM 1000 步 (FID 接近，10-20× 加速)
- DDIM 25 步 仍然可用 (轻微质量损失)
- DDIM 10 步 开始模糊 → LCM/Turbo 通过蒸馏解决这个问题

### 5.4 DDIM 与 latent space 的可控性

DDIM 确定性意味着 $x_T$ 和 $x_0$ 一一对应 → 可以做**潜空间插值** (image morphing): 找两张图各自的 $x_T$，在 latent 空间插值后采样回去。


In [ ]:
@torch.no_grad()
def ddim_sample(model, n_samples, image_size=28, n_steps=20, T=T_train, eta=0.0):
    """DDIM 采样
    
    eta=0.0: 完全确定性 (DDIM)
    eta=1.0: 退化为 DDPM
    """
    model.eval()
    x = torch.randn(n_samples, 1, image_size, image_size).to(device)
    
    # 选择子时间步序列: 均匀间隔
    step_seq = torch.linspace(0, T-1, n_steps + 1).long().tolist()
    step_seq = step_seq[::-1]  # 倒序: T-1 → 0
    
    for i in range(len(step_seq) - 1):
        t = step_seq[i]
        t_prev = step_seq[i + 1]
        
        t_batch = torch.full((n_samples,), t, device=device, dtype=torch.long)
        eps_pred = model(x, t_batch)
        
        alpha_bar_t = alphas_bar[t]
        alpha_bar_prev = alphas_bar[t_prev] if t_prev >= 0 else torch.tensor(1.0, device=device)
        
        # 预测 x_0
        x_0_pred = (x - (1 - alpha_bar_t).sqrt() * eps_pred) / alpha_bar_t.sqrt()
        
        # DDIM 更新公式
        sigma = eta * ((1 - alpha_bar_prev) / (1 - alpha_bar_t)).sqrt() * \
                (1 - alpha_bar_t / alpha_bar_prev).sqrt()
        
        noise = torch.randn_like(x) if eta > 0 and t > 0 else 0
        
        x = alpha_bar_prev.sqrt() * x_0_pred + \
            (1 - alpha_bar_prev - sigma**2).clamp(min=0).sqrt() * eps_pred + \
            sigma * noise
    
    model.train()
    return x


# 对比: DDPM 200 步 vs DDIM 不同步数
torch.manual_seed(0)
x_init_seed = 0

# 准备相同 init noise
def fixed_init(seed, n=4):
    g = torch.Generator(device='cpu').manual_seed(seed)
    return torch.randn(n, 1, 28, 28, generator=g).to(device)

results = {}

# DDPM 200 步
x_init = fixed_init(0, 4)
t0 = time.time()
@torch.no_grad()
def ddpm_sample_init(model, x_init, T=T_train):
    model.eval()
    x = x_init.clone()
    for t in reversed(range(T)):
        t_b = torch.full((x.size(0),), t, device=device, dtype=torch.long)
        eps_pred = model(x, t_b)
        alpha_t = alphas[t]; alpha_bar_t = alphas_bar[t]; beta_t = betas[t]
        x_mean = (1/alpha_t.sqrt()) * (x - (beta_t/(1-alpha_bar_t).sqrt()) * eps_pred)
        if t > 0:
            x = x_mean + beta_t.sqrt() * torch.randn_like(x)
        else:
            x = x_mean
    model.train()
    return x

results['DDPM 200步'] = (ddpm_sample_init(unet, x_init), time.time() - t0)

for n_steps in [50, 20, 10]:
    x_init = fixed_init(0, 4)
    t0 = time.time()
    out = ddim_sample(unet, 4, n_steps=n_steps)  # 注意: ddim 内部 init 是新随机的
    results[f'DDIM {n_steps}步'] = (out, time.time() - t0)

# 可视化
fig, axes = plt.subplots(len(results), 4, figsize=(8, 2*len(results)))
for i, (name, (imgs, t)) in enumerate(results.items()):
    for j in range(4):
        axes[i, j].imshow(imgs[j, 0].cpu().numpy(), cmap='gray', vmin=-1, vmax=1)
        axes[i, j].axis('off')
    axes[i, 0].set_ylabel(f'{name}\n{t:.2f}s', rotation=0, ha='right', va='center')
plt.suptitle('DDPM vs DDIM 采样步数对比', y=1.02)
plt.tight_layout()
plt.show()

for name, (_, t) in results.items():
    print(f"  {name:15s}: {t:.3f}s")
print("\n💡 步数减少 → 速度线性提升; 但 <10 步开始质量下降")


---
## 6. Classifier-Free Guidance (CFG) <a id='6'></a>

### 6.1 条件生成的两种范式

我们想生成「**符合条件 c (例如文本 prompt) 的图像**」。

| 方法 | 做法 | 缺点 |
|---|---|---|
| **Classifier Guidance** | 训练独立分类器 → 用梯度引导 | 需要额外训练分类器、对抗样本敏感 |
| **Classifier-Free Guidance (CFG)** | 同时训练有条件和无条件模型 | 仅多 ~10% 训练成本，效果更好 |

### 6.2 CFG 训练

每步以一定概率 (例如 10%) **丢弃条件**，让模型同时学会:
- 有条件预测: $\epsilon_\theta(x_t, t, c)$
- 无条件预测: $\epsilon_\theta(x_t, t, \emptyset)$

### 6.3 CFG 采样

采样时用线性外推:

$$\tilde\epsilon = \epsilon_\theta(x_t, t, \emptyset) + w \cdot (\epsilon_\theta(x_t, t, c) - \epsilon_\theta(x_t, t, \emptyset))$$

- $w = 0$: 退化为无条件生成
- $w = 1$: 普通条件生成
- $w > 1$ (例如 7.5): **强化条件方向**，更贴合 prompt 但多样性下降

### 6.4 Guidance Scale 的 trade-off

| w | 效果 |
|---|---|
| 1-3 | 多样性高，可能不太贴合 prompt |
| 5-10 | SD 默认范围 (7.5)，平衡 |
| >12 | 过饱和、伪影、对比度爆 |

**面试点**: 「CFG scale 的物理含义是什么？为什么大了会伪影？」
答: 数学上是从条件分布外推到「比条件还更条件」的方向，超出训练分布 → 走样。

### 6.5 实战示例 (代码演示，不依赖外部模型)

为了不下载 SD 大模型，下面用伪代码描述 CFG 在 SD pipeline 中的位置:

```python
# diffusers 库中的 SD CFG
text_emb = clip_encoder(prompt)         # 条件
uncond_emb = clip_encoder("")           # 无条件

for t in scheduler.timesteps:
    # 拼成 batch=2 一次前向算两个
    latent_input = torch.cat([latent, latent])
    cond_input = torch.cat([uncond_emb, text_emb])
    
    eps = unet(latent_input, t, cond_input)
    eps_uncond, eps_cond = eps.chunk(2)
    
    # CFG
    eps_cfg = eps_uncond + cfg_scale * (eps_cond - eps_uncond)
    
    latent = scheduler.step(eps_cfg, t, latent)
```

**注意**: CFG 让每步前向**变成 2 倍计算量** (cond + uncond)，但批量化后只是一次 batch=2 的前向。


---
## 7. Stable Diffusion 架构 <a id='7'></a>

### 7.1 SD = Latent Diffusion + Text Conditioning

**痛点**: 直接在像素空间 (512×512×3) 跑 Diffusion 太慢。

**解决**: SD 在 **latent 空间** (64×64×4) 跑 Diffusion，前后接 VAE encoder/decoder。

```
text prompt
    ↓ CLIP text encoder
text emb (77, 768)
    ↓
              [Cross-Attention 注入]
                       ↓
image (512×512×3)
    ↓ VAE encoder (8× 下采样)
latent (64×64×4)
    ↓ 加噪 (前向)
x_T (64×64×4)
    ↓ U-Net + Cross-Attn 反向去噪
x_0_latent (64×64×4)
    ↓ VAE decoder
image (512×512×3)
```

### 7.2 三大组件

1. **VAE**: 像素 ↔ latent 的双向映射 (KL 正则保证 latent 平滑)
2. **U-Net (核心)**: 在 latent 空间预测噪声
   - 多个 downsample / upsample stages
   - 每个 stage 有 ResBlock + Cross-Attention + Self-Attention
   - 文本通过 cross-attention 注入: Q 来自 latent, K/V 来自文本嵌入
3. **CLIP Text Encoder**: 把 prompt 编码为 (77, 768) 序列嵌入

### 7.3 SD 各代版本

| 版本 | 分辨率 | latent | text encoder | 参数 |
|---|---|---|---|---|
| SD 1.5 | 512 | 4ch 64×64 | CLIP-L (768) | 0.86B |
| SD 2.1 | 768 | 4ch 96×96 | OpenCLIP-H (1024) | 0.86B |
| SDXL | 1024 | 4ch 128×128 | CLIP-L + OpenCLIP-bigG (2048) | 2.6B + refiner 6.6B |
| SD 3 (MMDiT) | 1024 | 16ch | T5-XXL + CLIP | 8B |

### 7.4 关键工程权衡

- **latent 通道数**: 太少 (3) 重构差，太多 (16+) 训练慢；4 是甜蜜点
- **VAE 重构损失**: SD 的 VAE 不是无损的，文本/小细节会丢失 → SDXL 改进了 VAE
- **CFG scale 默认 7.5**: 是大量实验后的经验值


In [ ]:
# 用 mermaid 图概念性展示 SD 架构 (matplotlib 简化版)
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(0, 14); ax.set_ylim(0, 6); ax.axis('off')

def box(x, y, w, h, label, color='#E3F2FD'):
    ax.add_patch(plt.Rectangle((x, y), w, h, facecolor=color, edgecolor='#1976D2', lw=1.5))
    ax.text(x + w/2, y + h/2, label, ha='center', va='center', fontsize=10, fontweight='bold')

def arrow(x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

# Top row: prompt → CLIP
box(0.5, 4.5, 2, 1, 'Prompt\n"a cat..."', '#FFF9C4')
arrow(2.5, 5, 3.5, 5)
box(3.5, 4.5, 2, 1, 'CLIP Text\nEncoder', '#FFE0B2')
arrow(5.5, 5, 6.5, 5)
box(6.5, 4.5, 2, 1, 'Text Emb\n(77, 768)', '#FFF9C4')

# Middle: Latent diffusion
box(0.5, 2.5, 2, 1, 'Image\n(512×512×3)', '#C8E6C9')
arrow(2.5, 3, 3.5, 3)
box(3.5, 2.5, 2, 1, 'VAE Encoder\n(8× ↓)', '#FFCDD2')
arrow(5.5, 3, 6.5, 3)
box(6.5, 2.5, 2, 1, 'Latent\n(64×64×4)', '#C8E6C9')
arrow(8.5, 3, 9.5, 3)
box(9.5, 2.5, 3, 1, 'U-Net (×T)\nDDPM/DDIM', '#FFE0B2')

# Cross-attention arrow from text emb to U-Net
ax.annotate('', xy=(11, 3.5), xytext=(7.5, 4.5),
            arrowprops=dict(arrowstyle='->', color='red', lw=1.5, linestyle='--'))
ax.text(9, 4.2, 'cross-attn', color='red', fontsize=9)

# Bottom: VAE decode back
arrow(11, 2.5, 11, 1.5)
box(9.5, 0.5, 3, 1, 'VAE Decoder', '#FFCDD2')
arrow(9.5, 1, 8, 1)
box(6, 0.5, 2, 1, 'Image Out', '#C8E6C9')

ax.text(7, 5.7, 'Stable Diffusion 推理流程 (简化)', fontsize=14, ha='center', fontweight='bold')
plt.tight_layout()
plt.show()


---
## 8. ControlNet 原理 <a id='8'></a>

### 8.1 动机: 文本控制不够精准

用 prompt 「在房间中央有一只猫」无法精准控制猫的姿势、位置、大小。

ControlNet 让我们用**结构性条件** (canny 边缘 / depth / pose / segmentation) 控制生成。

### 8.2 ControlNet 架构 (这是面试硬点)

**核心思想**: 复制 SD U-Net 的 encoder 部分，作为可训练副本；通过**零卷积**和原 U-Net 相加。

```
输入条件 (canny/depth/...)
        ↓
    [可训练副本]                    [冻结的原 SD U-Net]
        ↓ 零卷积                              ↓
    跳跃连接 (与原 decoder 残差相加)         decoder 输出
                                              ↓
                                          噪声预测
```

### 8.3 关键设计: 零卷积 (Zero Convolution)

副本和原模型相加的位置用 **1×1 conv with weights initialized to 0**:
- 训练开始时: 输出 = 原 SD 输出 (因为副本贡献 = 0)
- 训练过程: 零卷积权重逐渐学到非零值，副本逐渐贡献
- **保护预训练权重**: 即使条件不准也不会破坏 SD 的能力

这就是为什么 ControlNet **小数据集 (10K-100K) 也能训练好** — 起点就是已经能用的 SD。

### 8.4 ControlNet 与你的博士论文

你的论文「Ontology-guided synthetic data」用的就是这个工具链:
- **Ontology (OWL)** → 结构化生成 prompt + ControlNet 条件
- **prompt → SD/ControlNet** → 合成图像
- **CLIP Score 筛选** → 过滤低质量
- **混合训练 ResNet/ViT** → 验证下游有效性

今天 D5 是工具层面，**明天 D6 你要把这一切整合成端到端 Pipeline**。

### 8.5 ControlNet 变体

| 变体 | 条件输入 | 适用场景 |
|---|---|---|
| Canny | 边缘图 | 保持线稿结构 |
| Depth | 深度图 | 3D 空间布局 |
| Pose (OpenPose) | 骨架 | 人物姿势 |
| Segmentation | 语义图 | 场景布局 |
| Scribble | 手绘 | 创意控制 |
| MLSD | 直线 | **建筑/施工场景 — 你的方向！** |

**面试 talk track**: 「我用 MLSD ControlNet 控制建筑物结构线条，配合本体生成的施工要素描述，能稳定合成符合规范的施工场景图像，FID 和 CLIP Score 验证效果...」

### 8.6 SD + ControlNet 推理伪代码

```python
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
import cv2

# 加载
controlnet = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-canny")
pipe = StableDiffusionControlNetPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", 
                                                          controlnet=controlnet)

# 准备条件: canny 边缘
image = cv2.imread("scaffold.jpg")
canny = cv2.Canny(image, 100, 200)
canny_image = Image.fromarray(canny[..., None].repeat(3, -1))

# 生成
out = pipe(
    prompt="a construction worker on scaffolding, safety helmet, high quality",
    image=canny_image,
    num_inference_steps=30,
    guidance_scale=7.5,
    controlnet_conditioning_scale=1.0  # ControlNet 强度
).images[0]
```


---
## 9. 评估指标: FID & CLIP Score <a id='9'></a>

### 9.1 FID (Fréchet Inception Distance)

衡量两个图像分布的距离 (生成 vs 真实):

$$\text{FID} = \| \mu_r - \mu_g \|^2 + \text{Tr}(\Sigma_r + \Sigma_g - 2\sqrt{\Sigma_r \Sigma_g})$$

其中 $\mu, \Sigma$ 是 Inception-v3 倒数第二层特征的均值和协方差。

**特性**:
- 越低越好 (0 = 分布完全一致)
- SD 1.5 在 COCO val 上 FID ≈ 12, GAN 时代 ~20+
- **小样本 (<2000) 不可靠**: 协方差估计不稳定

### 9.2 CLIP Score

衡量生成图像和文本 prompt 的对齐程度:

$$\text{CLIP Score} = \cos(\text{CLIP}_{\text{img}}(x), \text{CLIP}_{\text{text}}(c)) \times 100$$

**特性**:
- 越高越好 (理论上 100，实际 SD ~25-35)
- 对**多样性盲**: 如果模型只生成一种符合 prompt 的图也得高分
- **类别外**: 训练集没见过的 prompt (例如某些专业术语) 不可靠

### 9.3 双指标互补

| | FID 高 | FID 低 |
|---|---|---|
| **CLIP Score 高** | 贴合 prompt 但分布偏 | 理想 |
| **CLIP Score 低** | 模型烂 | 多样性好但不听话 |

### 9.4 你的合成数据评估流程 (与你论文方法一致)

```
1. 生成 N 张 (本体描述 → SD/ControlNet)
2. CLIP Score 筛选: 保留 score > τ 的样本
3. (可选) FID(合成, 真实少量) 评估分布距离
4. 混合训练 ResNet/ViT
5. 下游任务指标 (mAP / accuracy) 才是终极评估
```

### 9.5 其他常见指标

- **IS (Inception Score)**: 老指标，衡量生成多样性 + 清晰度，已基本被 FID 取代
- **LPIPS**: 衡量两张图的感知距离 (用 VGG/AlexNet 特征)，常用于编辑/恢复任务
- **DreamSim / DINOv2 features**: 新一代感知距离指标


In [ ]:
# CLIP Score 简化实现 (无需下载 CLIP)
# 这里用一个 mock 的「CLIP-like」函数演示流程

def mock_clip_score(images, texts):
    """Mock CLIP Score for demonstration"""
    # 真实实现:
    #   img_features = clip.encode_image(images)
    #   text_features = clip.encode_text(texts)
    #   return F.cosine_similarity(img_features, text_features) * 100
    
    # 这里返回模拟值 (面试时引用代码思路即可)
    n = len(images) if hasattr(images, '__len__') else 1
    return torch.rand(n) * 30 + 5  # 模拟 5-35 范围


# 演示「合成数据筛选」工作流
print("模拟合成数据筛选流程:")
print("=" * 60)
n_generated = 100
mock_images = list(range(n_generated))
mock_prompts = ["a worker on scaffolding"] * n_generated

scores = mock_clip_score(mock_images, mock_prompts)

threshold_pct = 50  # 保留前 50%
threshold = torch.quantile(scores, threshold_pct / 100)
keep_mask = scores >= threshold

print(f"  生成: {n_generated}")
print(f"  CLIP Score 分布: mean={scores.mean():.2f}, std={scores.std():.2f}, "
      f"min={scores.min():.2f}, max={scores.max():.2f}")
print(f"  阈值 (top {100-threshold_pct}%): {threshold:.2f}")
print(f"  保留: {keep_mask.sum().item()}")
print(f"  过滤掉: {(~keep_mask).sum().item()}")

# 可视化
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(scores.numpy(), bins=20, alpha=0.7, color='#1976D2')
ax.axvline(threshold.item(), color='red', linestyle='--', label=f'threshold={threshold:.2f}')
ax.set_xlabel('CLIP Score')
ax.set_ylabel('Count')
ax.set_title(f'合成数据 CLIP Score 分布 (筛选阈值)')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# FID 计算的简化展示 (用低维特征模拟)
from scipy.linalg import sqrtm

def fid_score(features_real, features_gen):
    """简化 FID: 给定两组特征向量，计算 Fréchet 距离"""
    mu_r = features_real.mean(axis=0)
    mu_g = features_gen.mean(axis=0)
    sigma_r = np.cov(features_real, rowvar=False)
    sigma_g = np.cov(features_gen, rowvar=False)
    
    diff = mu_r - mu_g
    covmean = sqrtm(sigma_r @ sigma_g)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    
    fid = diff @ diff + np.trace(sigma_r + sigma_g - 2 * covmean)
    return fid


# 模拟: 真实 vs 生成 (close vs far)
np.random.seed(42)
n = 1000
d = 64  # 用低维代替 Inception 2048

# 真实分布: N(0, I)
real = np.random.randn(n, d)

# 生成分布 1: 与真实一致
gen_close = np.random.randn(n, d)
# 生成分布 2: 偏移
gen_far = np.random.randn(n, d) + 0.5
# 生成分布 3: 方差缩小 (mode collapse)
gen_collapse = np.random.randn(n, d) * 0.3

print("FID 计算演示 (越低越好):")
print(f"  真实 vs 真实 (理论 0):     {fid_score(real, np.random.randn(n, d)):.3f}")
print(f"  真实 vs 同分布 gen:        {fid_score(real, gen_close):.3f}")
print(f"  真实 vs 偏移 gen:          {fid_score(real, gen_far):.3f}")
print(f"  真实 vs mode collapse gen: {fid_score(real, gen_collapse):.3f}")

# 小样本不稳定演示
print("\n💡 FID 在小样本上不稳定:")
for n_small in [50, 200, 1000, 5000]:
    fids = []
    for _ in range(5):
        a = np.random.randn(n_small, d)
        b = np.random.randn(n_small, d)
        fids.append(fid_score(a, b))
    print(f"  n={n_small}: FID(同分布) = {np.mean(fids):.3f} ± {np.std(fids):.3f}")

print("\n💡 真实场景: 至少 2000 样本; 不同模型对比要用同 reference set")


---
## 10. 总结 & 思考题 <a id='10'></a>

### 今日核心知识点

1. **DDPM 前向**: $x_t = \sqrt{\bar\alpha_t} x_0 + \sqrt{1-\bar\alpha_t} \epsilon$，一步到位
2. **DDPM 训练**: 预测噪声 ε，loss 是简单 MSE
3. **DDIM**: 确定性 + 跳步采样，10-20× 加速
4. **CFG**: 同时训练有/无条件，采样时外推 — 用 CFG scale 控制贴合度
5. **SD = Latent Diffusion + CLIP Text Cond + Cross-Attn**: 在 latent 空间扩散，省 64× 计算
6. **ControlNet**: 副本 + 零卷积，结构性条件控制
7. **评估**: FID (分布距离) + CLIP Score (文图对齐) 互补

### 面试高频问题

1. **DDPM 为什么预测 ε 而不是 x_0？** ε 是标准正态、量纲稳定，loss 数值更稳；理论上等价
2. **DDIM 为什么能跳步？** 训练目标只决定噪声预测，采样过程 (反向 SDE/ODE) 可以自由设计
3. **CFG scale 大了为什么伪影？** 数学上是分布外推，超出训练数据支持的 manifold
4. **ControlNet 零卷积的物理意义？** 起点是「不影响原模型」，保护预训练权重，让小数据集训练成为可能
5. **SD 在 latent 空间扩散的好处？** 64× 计算量 (8×8 空间下采样) + VAE 已经压缩了感知冗余
6. **FID 不可靠的场景？** 小样本 (<2000)、协方差估计不稳；以及 Inception 训于 ImageNet 对非自然图像 (医学/施工) 不准
7. **你的合成数据 domain gap 怎么解决？** ControlNet 注入真实结构、CLIP Score 严筛、混合训练比例消融、(可选) test-time augmentation

### 与你博士论文的连接

- 你过去用过 SD/ControlNet，但**今天补齐了原理层**:
  - 知道为什么 ControlNet 能控制结构 (零卷积保护预训练)
  - 知道为什么 CFG scale=7.5 是甜蜜点
  - 知道为什么合成数据筛选用 CLIP Score 而不是 FID (你的样本量小)
- 这些点是论文里没写但**面试官会追问**的「为什么」，今天补上后讲故事的密度上一个台阶

### 明日预习: 合成数据 Pipeline v1 ★ — 锚点项目核心

- 端到端整合: 本体描述 → ControlNet 生成 → CLIP 筛选 → 混合训练 → ResNet/ViT 对比 baseline
- **明天就是把今天学的全部用起来**


In [ ]:
print("=" * 60)
print("W2 Day5 完成! 🎉")
print("=" * 60)
print("""
今日成果:
  ✅ DDPM 前向加噪公式 + linear/cosine schedule 对比可视化
  ✅ 训练目标推导 (噪声预测 MSE) + 采样公式
  ✅ TinyUNet + 合成数据集训练完整 DDPM
  ✅ DDPM 反向去噪可视化 (T → 0)
  ✅ DDIM 实现 + 步数对比 (200 vs 50 vs 20 vs 10)
  ✅ Classifier-Free Guidance 数学 + SD 中的位置
  ✅ Stable Diffusion 架构图 (VAE + U-Net + CLIP)
  ✅ ControlNet 零卷积原理 + MLSD 应用 talk track
  ✅ CLIP Score 筛选工作流 + FID 小样本陷阱

明天 D6 锚点项目 ★ 准备清单:
  □ 确定本体: 你已有 ConSE 本体 → 选 5-10 个施工场景类
  □ 准备 ControlNet 条件图: 用真实图的 canny/MLSD 生成
  □ 准备 prompt 模板: "a construction site with [entity], [activity], ..."
  □ 设置筛选阈值: CLIP Score > 25 (典型值)
  □ 混合比例消融: 100% 真 / 50:50 / 30:70 / 100% 合成
""")